In [ ]:
from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F
import matplotlib.pyplot as plt

session = get_active_session()
df = session.sql("""
    SELECT p.Details:category::STRING as Category, SUM(o.Amount) as Total_Sales
    FROM STYLEHUB_DB.PUBLIC.ORDERS o JOIN STYLEHUB_DB.PUBLIC.PRODUCTS p ON o.ProductID = p.ProductID
    GROUP BY 1
""").to_pandas()

df.plot(kind='bar', x='CATEGORY', y='TOTAL_SALES', color='orange')
plt.title('Sales Category Breakdown')
plt.ylabel('Revenue (INR)')
plt.show()


In [ ]:
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()
reviews_df = session.table("STYLEHUB_DB.PUBLIC.PRODUCT_REVIEWS").to_pandas()

def calculate_sentiment(text):
    if any(word in text.lower() for word in ['worst', 'disappointing', 'flat', 'stopped working']):
        return "Negative 👎"
    elif any(word in text.lower() for word in ['fire', 'comfortable', 'premium', 'recommended', 'good', 'enough space']):
        return "Positive 👍"
    else:f
     return "Neutral 😐"

reviews_df['SENTIMENT_RESULT'] = reviews_df['REVIEWTEXT'].apply(calculate_sentiment)

print(reviews_df[['REVIEWTEXT', 'SENTIMENT_RESULT']])

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from collections import Counter
import re
import numpy as np
from snowflake.snowpark.context import get_active_session

session = get_active_session()
df_reviews = session.table("STYLEHUB_DB.PUBLIC.PRODUCT_REVIEWS").to_pandas()

all_words = []
for review in df_reviews['REVIEWTEXT']:
    clean_text = re.sub(r'[^\w\s]', '', review.lower())
    words = clean_text.split()
    all_words.extend(words)

stop_words = {'this', 'is', 'for', 'the', 'and', 'to', 'in', 'it', 'has', 'my', 'with', 'but', 'a', 'of'}
filtered_words = [w for w in all_words if w not in stop_words]

word_counts = Counter(filtered_words)
df_words = pd.DataFrame(word_counts.most_common(10), columns=['Word', 'Frequency'])
df_words = df_words.sort_values('Frequency', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#1a1a2e')

colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(df_words)))
bars = ax.barh(df_words['Word'], df_words['Frequency'], color=colors, edgecolor='white', linewidth=0.5, height=0.7)

for bar in bars:
    bar.set_alpha(0.9)
    width = bar.get_width()
    ax.text(width + 0.1, bar.get_y() + bar.get_height()/2, f'{int(width)}',
            va='center', ha='left', color='white', fontsize=11, fontweight='bold')

ax.set_title('Top Buzzwords in StyleHub Reviews', fontsize=16, fontweight='bold',
             color='white', pad=20)
ax.set_xlabel('Frequency', fontsize=12, color='#aaaaaa')
ax.set_ylabel('')
ax.tick_params(colors='white', labelsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_color('#444444')
ax.spines['left'].set_color('#444444')
ax.xaxis.grid(True, alpha=0.15, color='white')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from snowflake.snowpark.context import get_active_session

session = get_active_session()
df = session.sql("SELECT CITY, SUM(TOTAL_REVENUE) as REVENUE FROM STYLEHUB_DB.PUBLIC.V_SALES_ANALYTICS GROUP BY 1").to_pandas()

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(df["CITY"], df["REVENUE"])
ax.set_xlabel("City")
ax.set_ylabel("Revenue")
ax.set_title("Revenue by City")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print("Raw Sales Data")
df